## 1️⃣ Google Drive'ı Bağla

In [ ]:
from google.colab import drive
import os

# Google Drive'ı mount et
drive.mount('/content/drive')

print("\n✅ Google Drive bağlandı!")
print("📁 Drive konumu: /content/drive/MyDrive/")

## 2️⃣ GPU Kontrolü ve Ortam Bilgileri

In [ ]:
import torch
import locale

# Sistem bilgileri
def get_system_info():
    print("="*70)
    print("🖥️  SİSTEM BİLGİLERİ")
    print("="*70)
    
    # GPU kontrolü
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"\n✅ GPU: {gpu_name}")
        print(f"   VRAM: {gpu_memory:.1f} GB")
        print(f"   CUDA Version: {torch.version.cuda}")
        print(f"   PyTorch Version: {torch.__version__}")
        
        # T4 GPU için özel öneriler
        if 'T4' in gpu_name:
            print("\n💡 T4 GPU Tespit Edildi - Optimal Ayarlar:")
            print("   • Önerilen Batch Size: 32-64")
            print("   • Mixed Precision (AMP): Aktif")
            print("   • Tahmini Eğitim Süresi: 1-2 saat")
    else:
        print("\n⚠️  GPU BULUNAMADI!")
        print("   Lütfen: Runtime > Change runtime type > GPU (T4) seçin")
        return False
    
    # RAM bilgisi
    import psutil
    ram = psutil.virtual_memory()
    print(f"\n💾 RAM: {ram.total / 1024**3:.1f} GB")
    print(f"   Kullanılabilir: {ram.available / 1024**3:.1f} GB")
    
    print("\n" + "="*70)
    return True

# Sistem bilgilerini göster
if not get_system_info():
    raise RuntimeError("GPU bulunamadı! Lütfen GPU runtime seçin.")

## 3️⃣ Gerekli Paketleri Yükle

In [ ]:
%%capture
# Ultralytics YOLO paketini yükle (çıktıları gizle)
!pip install -q ultralytics

print("✅ Ultralytics YOLO yüklendi")

In [ ]:
# Gerekli kütüphaneleri import et
from ultralytics import YOLO
import yaml
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter, defaultdict
import shutil
import zipfile
from IPython.display import display, HTML
import cv2

# Görsel ayarları
plt.rcParams['figure.figsize'] = (15, 8)
sns.set_style("whitegrid")

print("✅ Tüm kütüphaneler yüklendi")

## 4️⃣ Veri Setini Yükle ve Unzip Et

### 📝 ÖNEMLİ:
Veri setinizi (dataset_cleaned.zip) Google Drive'ınıza yükleyin:
- **Konum:** `MyDrive/datasets/dataset_cleaned.zip`
- veya aşağıdaki hücrede kendi yolunuzu belirtin

In [ ]:
# Veri seti zip dosyasının konumu (kendi yolunuzu girin)
ZIP_PATH = '/content/drive/MyDrive/datasets/dataset_cleaned.zip'  # 👈 BURAYA KENDİ YOLUNUZU YAZIN

# Unzip edilecek konum
EXTRACT_PATH = '/content/dataset_cleaned'

print("="*70)
print("📦 VERİ SETİ YÜKLEME")
print("="*70)

# Zip dosyasının varlığını kontrol et
if not os.path.exists(ZIP_PATH):
    print(f"\n❌ HATA: Zip dosyası bulunamadı!")
    print(f"   Beklenen konum: {ZIP_PATH}")
    print(f"\n💡 Çözüm:")
    print(f"   1. dataset_cleaned.zip dosyasını Google Drive'a yükleyin")
    print(f"   2. Yukarıdaki ZIP_PATH değişkenini güncelleyin")
    raise FileNotFoundError(f"Zip dosyası bulunamadı: {ZIP_PATH}")

print(f"\n✅ Zip dosyası bulundu: {ZIP_PATH}")

# Zip dosyasının boyutunu göster
zip_size = os.path.getsize(ZIP_PATH) / (1024**2)
print(f"   Boyut: {zip_size:.1f} MB")

# Eski klasörü temizle (varsa)
if os.path.exists(EXTRACT_PATH):
    print(f"\n🗑️  Eski klasör temizleniyor...")
    shutil.rmtree(EXTRACT_PATH)

# Unzip işlemi
print(f"\n📦 Unzip ediliyor...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print(f"✅ Unzip tamamlandı: {EXTRACT_PATH}")

# İçeriği kontrol et
dataset_path = Path(EXTRACT_PATH)
if dataset_path.exists():
    subdirs = [d.name for d in dataset_path.iterdir() if d.is_dir()]
    print(f"\n📂 Klasör içeriği: {subdirs}")
    
    # Train/valid/test sayıları
    for split in ['train', 'valid', 'test']:
        img_dir = dataset_path / split / 'images'
        if img_dir.exists():
            img_count = len(list(img_dir.glob('*')))
            print(f"   • {split}: {img_count} görsel")
else:
    raise RuntimeError(f"Dataset klasörü bulunamadı: {EXTRACT_PATH}")

print("\n" + "="*70)

## 5️⃣ data.yaml Dosyasını Güncelle

In [ ]:
# data.yaml dosyasını oku ve güncelle
data_yaml_path = dataset_path / 'data.yaml'

print("📝 data.yaml güncelleniyor...")

if data_yaml_path.exists():
    with open(data_yaml_path, 'r', encoding='utf-8') as f:
        data_config = yaml.safe_load(f)
    
    # Path'i Colab için güncelle
    data_config['path'] = str(dataset_path)
    data_config['train'] = 'train/images'
    data_config['val'] = 'valid/images'
    data_config['test'] = 'test/images'
    
    # Güncellenmiş dosyayı kaydet
    with open(data_yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_config, f, default_flow_style=False, allow_unicode=True)
    
    print(f"\n✅ data.yaml güncellendi")
    print(f"\n📊 Veri Seti Bilgileri:")
    print(f"   • Path: {data_config['path']}")
    print(f"   • Sınıf Sayısı: {data_config['nc']}")
    print(f"   • Train: {data_config['train']}")
    print(f"   • Valid: {data_config['val']}")
    print(f"   • Test: {data_config['test']}")
else:
    raise FileNotFoundError(f"data.yaml bulunamadı: {data_yaml_path}")

## 6️⃣ Veri Dengesizliği Analizi

In [ ]:
def parse_yolo_labels(label_file):
    """YOLO format etiket dosyasını parse et"""
    classes = []
    with open(label_file, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                class_id = int(parts[0])
                classes.append(class_id)
    return classes

# Train setindeki sınıf dağılımını analiz et
print("="*70)
print("⚖️  VERİ DENGESİZLİĞİ ANALİZİ")
print("="*70)

train_labels_dir = dataset_path / 'train' / 'labels'
class_counts = Counter()

if train_labels_dir.exists():
    label_files = list(train_labels_dir.glob('*.txt'))
    for label_file in label_files:
        classes = parse_yolo_labels(label_file)
        for cls in classes:
            class_counts[cls] += 1

# İstatistikler
total_samples = sum(class_counts.values())
num_classes = data_config['nc']
unique_classes = len(class_counts)

print(f"\n📊 Sınıf Dağılımı:")
print(f"   • Toplam annotation: {total_samples}")
print(f"   • Kullanılan sınıf: {unique_classes} / {num_classes}")
print(f"   • En yaygın sınıf: {max(class_counts.values())} örnek")
print(f"   • En nadir sınıf: {min(class_counts.values())} örnek")
print(f"   • Dengesizlik oranı: {max(class_counts.values()) / min(class_counts.values()):.1f}x")

# Class weights hesapla
class_weights = {}
for cls_id in range(num_classes):
    count = class_counts.get(cls_id, 1)
    weight = total_samples / (num_classes * count)
    class_weights[cls_id] = weight

# Ağırlıkları normalize et ve sınırla
weights_array = np.array([class_weights[i] for i in range(num_classes)])
weights_array = weights_array / weights_array.mean()
weights_array = np.clip(weights_array, 0.5, 10.0)

print(f"\n⚖️  Class Weights:")
print(f"   • Min weight: {weights_array.min():.2f}")
print(f"   • Max weight: {weights_array.max():.2f}")
print(f"   • Mean weight: {weights_array.mean():.2f}")

print("\n✅ Veri dengesizliği için çözümler uygulanacak:")
print("   • Class loss gain artırıldı (0.5 → 1.0)")
print("   • MixUp augmentation eklendi (15%)")
print("   • Copy-Paste augmentation eklendi (30%)")
print("   • Güçlendirilmiş geometrik augmentations")

print("\n" + "="*70)

## 7️⃣ Sınıf Dağılımı Görselleştirmesi

In [ ]:
# En yaygın 20 sınıfı görselleştir
top_20 = dict(class_counts.most_common(20))
class_names = data_config.get('names', [])

classes = [class_names[cls] if cls < len(class_names) else f"C{cls}" for cls in top_20.keys()]
counts = list(top_20.values())

# İsimleri kısalt
classes = [c[:30] + '...' if len(c) > 30 else c for c in classes]

plt.figure(figsize=(12, 8))
bars = plt.barh(classes, counts, color='steelblue')
plt.xlabel('Örnek Sayısı', fontsize=12)
plt.title('Train Set - En Yaygın 20 Sınıf', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

# Değerleri ekle
for bar in bars:
    width = bar.get_width()
    plt.text(width, bar.get_y() + bar.get_height()/2, 
            f'{int(width)}', ha='left', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("📊 Sınıf dağılımı görselleştirildi")

## 8️⃣ YOLOv8 Modelini Yükle

In [ ]:
print("="*70)
print("📦 MODEL YÜKLEME")
print("="*70)

# YOLOv8n modelini yükle
model = YOLO('yolov8n.pt')

print("\n✅ YOLOv8n pretrained model yüklendi")
print("   Model: yolov8n.pt (nano - hızlı ve verimli)")
print("   Parameters: ~3.2M")
print("   Model Size: ~6 MB")

print("\n" + "="*70)

## 9️⃣ Eğitim Parametreleri (Veri Dengesizliği Çözümlü)

In [ ]:
print("="*70)
print("⚙️  EĞİTİM PARAMETRELERİ (T4 GPU Optimizasyonlu)")
print("="*70)

# T4 GPU için optimize edilmiş parametreler
EPOCHS = 100
BATCH_SIZE = 32       # T4 için optimal (16GB VRAM)
IMG_SIZE = 640
PATIENCE = 50
WORKERS = 8           # Colab için optimal

print(f"\n📋 Temel Ayarlar:")
print(f"   • Epochs: {EPOCHS}")
print(f"   • Batch Size: {BATCH_SIZE} (T4 optimized)")
print(f"   • Image Size: {IMG_SIZE}")
print(f"   • Patience: {PATIENCE}")
print(f"   • Workers: {WORKERS}")
print(f"   • Device: GPU (T4)")
print(f"   • Mixed Precision: Aktif")

print(f"\n🎨 Veri Dengesizliği Çözümleri:")
print(f"   • Class Loss Gain: 1.0 (artırıldı)")
print(f"   • MixUp: 15%")
print(f"   • Copy-Paste: 30%")
print(f"   • Mosaic: 100%")
print(f"   • Rotation: ±10°")
print(f"   • Shear: 5°")
print(f"   • Scale: 0.7")

print(f"\n⏱️  Tahmini Süre: 1-2 saat")
print(f"💾 Sonuçlar: /content/runs/detect/train/")

print("\n" + "="*70)

## 🔟 EĞİTİMİ BAŞLAT 🚀

### ⚠️ UYARI:
- Eğitim başladıktan sonra tarayıcıyı kapatmayın
- Colab bağlantısı kopmamalı
- ~1-2 saat sürecek

### 💡 İPUCU:
Colab Pro kullanıyorsanız background execution aktiftir

In [ ]:
print("="*70)
print("🚀 EĞİTİM BAŞLIYOR...")
print("="*70)
print(f"\n⏱️  Başlangıç Zamanı: {pd.Timestamp.now()}")
print("\n📊 Eğitim ilerlemesi aşağıda görüntülenecek...\n")

# Modeli eğit
results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=0,                   # GPU
    patience=PATIENCE,
    save=True,
    project='/content/runs/detect',
    name='train',
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    verbose=True,
    seed=42,
    deterministic=True,
    single_cls=False,
    rect=False,
    cos_lr=False,
    close_mosaic=15,
    resume=False,
    amp=True,                   # Mixed Precision
    fraction=1.0,
    profile=False,
    freeze=None,
    
    # Learning rate
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # Loss gains - VERİ DENGESİZLİĞİ İÇİN AYARLANDI
    box=7.5,
    cls=1.0,                    # Class loss gain ARTIRILDI (0.5 → 1.0)
    dfl=1.5,
    
    # Data augmentation - GÜÇLENDİRİLDİ
    hsv_h=0.025,                # HSV-Hue
    hsv_s=0.8,                  # HSV-Saturation
    hsv_v=0.5,                  # HSV-Value
    degrees=10.0,               # Rotation
    translate=0.15,             # Translation
    scale=0.7,                  # Scale
    shear=5.0,                  # Shear
    perspective=0.0005,         # Perspective
    flipud=0.0,                 # Vertical flip
    fliplr=0.5,                 # Horizontal flip
    mosaic=1.0,                 # Mosaic
    mixup=0.15,                 # MixUp - NADİR SINIFLAR İÇİN
    copy_paste=0.3,             # Copy-paste - NADİR SINIFLAR İÇİN
    auto_augment='randaugment',
    erasing=0.5,                # Random erasing
    crop_fraction=1.0,
    
    # Diğer
    label_smoothing=0.0,
    nbs=64,
    workers=WORKERS,
    plots=True,                 # Grafikleri kaydet
    val=True                    # Validation yap
)

print("\n" + "="*70)
print("✅ EĞİTİM TAMAMLANDI!")
print("="*70)
print(f"\n⏱️  Bitiş Zamanı: {pd.Timestamp.now()}")
print(f"\n💾 Sonuçlar: /content/runs/detect/train/")

## 1️⃣1️⃣ Eğitim Sonuçlarını Görselleştir

In [ ]:
print("="*70)
print("📊 EĞİTİM SONUÇLARI")
print("="*70)

results_dir = Path('/content/runs/detect/train')

# 1. Training Results
results_img = results_dir / 'results.png'
if results_img.exists():
    print("\n📈 Training Metrics:")
    img = Image.open(results_img)
    plt.figure(figsize=(20, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Results - Loss & Metrics', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠️  results.png bulunamadı")

In [ ]:
# 2. Confusion Matrix
confusion_matrix = results_dir / 'confusion_matrix.png'
if confusion_matrix.exists():
    print("\n🎯 Confusion Matrix:")
    img = Image.open(confusion_matrix)
    plt.figure(figsize=(14, 12))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix - Sınıf Performansı', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Confusion Matrix İpuçları:")
    print("   • Diagonal'daki koyu renkler doğru tahminleri gösterir")
    print("   • Off-diagonal değerler karışıklıkları gösterir")
    print("   • Nadir sınıfların performansına dikkat edin")
else:
    print("\n⚠️  confusion_matrix.png bulunamadı")

In [ ]:
# 3. F1 Curve
f1_curve = results_dir / 'F1_curve.png'
if f1_curve.exists():
    print("\n📊 F1 Score Curve:")
    img = Image.open(f1_curve)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('F1-Confidence Curve', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# 4. Precision-Recall Curve
pr_curve = results_dir / 'PR_curve.png'
if pr_curve.exists():
    print("\n📊 Precision-Recall Curve:")
    img = Image.open(pr_curve)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

In [ ]:
# 5. Örnek Tahmin Görselleri
val_batch_labels = results_dir / 'val_batch0_labels.jpg'
val_batch_pred = results_dir / 'val_batch0_pred.jpg'

if val_batch_labels.exists() and val_batch_pred.exists():
    print("\n🎨 Örnek Tahminler (Val Batch):")
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # Ground Truth
    img1 = Image.open(val_batch_labels)
    axes[0].imshow(img1)
    axes[0].axis('off')
    axes[0].set_title('Ground Truth Labels', fontsize=14, fontweight='bold')
    
    # Predictions
    img2 = Image.open(val_batch_pred)
    axes[1].imshow(img2)
    axes[1].axis('off')
    axes[1].set_title('Model Predictions', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 1️⃣2️⃣ Model Performans Metrikleri

In [ ]:
# Validation metriklerini yükle
best_model_path = results_dir / 'weights' / 'best.pt'

if best_model_path.exists():
    print("="*70)
    print("🎯 FINAL MODEL PERFORMANSI")
    print("="*70)
    
    # En iyi modeli yükle
    best_model = YOLO(str(best_model_path))
    
    # Validation yap
    print("\n📊 Validation çalıştırılıyor...\n")
    metrics = best_model.val()
    
    print("\n" + "="*70)
    print("📊 PERFORMANS METRİKLERİ")
    print("="*70)
    
    # Ana metrikler
    print(f"\n🎯 Detection Metrikleri:")
    print(f"   • mAP50:     {metrics.box.map50:.4f}")
    print(f"   • mAP50-95:  {metrics.box.map:.4f}")
    print(f"   • Precision: {metrics.box.mp:.4f}")
    print(f"   • Recall:    {metrics.box.mr:.4f}")
    
    # Performans değerlendirmesi
    print(f"\n📈 Değerlendirme:")
    
    if metrics.box.map50 > 0.85:
        print("   ✅ MÜKEMMEL! Model çok iyi performans gösteriyor")
    elif metrics.box.map50 > 0.70:
        print("   ✅ ÇOK İYİ! Model iyi performans gösteriyor")
    elif metrics.box.map50 > 0.50:
        print("   ⚠️  İYİ: Model kabul edilebilir performans gösteriyor")
    else:
        print("   ⚠️  DÜŞÜK: Model performansı artırılmalı")
    
    print(f"\n💡 İyileştirme Önerileri:")
    if metrics.box.map50 < 0.70:
        print("   • Daha fazla epoch deneyin (150-200)")
        print("   • Learning rate'i ayarlayın")
        print("   • Daha büyük model deneyin (yolov8s veya yolov8m)")
    
    if metrics.box.mp < 0.80:
        print("   • False positive'leri azaltmak için confidence threshold artırın")
    
    if metrics.box.mr < 0.75:
        print("   • Nadir sınıflar için daha fazla veri toplayın")
        print("   • Augmentation'ı daha da güçlendirin")
    
    print("\n" + "="*70)
else:
    print("⚠️  Best model bulunamadı")

## 1️⃣3️⃣ Test Seti Üzerinde Tahmin

In [ ]:
# Test setinde rastgele görseller seç ve tahmin yap
test_images_dir = dataset_path / 'test' / 'images'

if test_images_dir.exists() and best_model_path.exists():
    print("="*70)
    print("🧪 TEST SETİ TAHMİNLERİ")
    print("="*70)
    
    # Rastgele 6 görsel seç
    import random
    test_images = list(test_images_dir.glob('*.jpg')) + list(test_images_dir.glob('*.png'))
    sample_images = random.sample(test_images, min(6, len(test_images)))
    
    # Tahmin yap
    results = best_model.predict(sample_images, conf=0.25, iou=0.45, imgsz=640)
    
    # Görselleri görselleştir
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for idx, (img_path, result) in enumerate(zip(sample_images, results)):
        # Görsel üzerine tahminleri çiz
        img_with_boxes = result.plot()
        img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(img_rgb)
        axes[idx].axis('off')
        axes[idx].set_title(f'{img_path.name}\nDetections: {len(result.boxes)}', 
                          fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n✅ {len(sample_images)} test görseli üzerinde tahmin yapıldı")
else:
    print("⚠️  Test görselleri veya model bulunamadı")

## 1️⃣4️⃣ Modeli Export Et (ONNX, TorchScript)

In [ ]:
if best_model_path.exists():
    print("="*70)
    print("📦 MODEL EXPORT")
    print("="*70)
    
    # ONNX formatına export
    print("\n🔄 ONNX formatına export ediliyor...")
    best_model.export(format='onnx', imgsz=640, simplify=True)
    print("✅ ONNX export tamamlandı")
    
    # TorchScript formatına export
    print("\n🔄 TorchScript formatına export ediliyor...")
    best_model.export(format='torchscript', imgsz=640)
    print("✅ TorchScript export tamamlandı")
    
    print("\n💡 Export edilen modeller:")
    weights_dir = results_dir / 'weights'
    exported_files = list(weights_dir.glob('*'))
    for file in exported_files:
        size_mb = file.stat().st_size / (1024**2)
        print(f"   • {file.name:30s} ({size_mb:.1f} MB)")
    
    print("\n" + "="*70)
else:
    print("⚠️  Model dosyası bulunamadı")

## 1️⃣5️⃣ Sonuçları Google Drive'a Kaydet

In [ ]:
# Sonuçları Google Drive'a kopyala
SAVE_PATH = '/content/drive/MyDrive/yolov8_training_results'  # 👈 İSTERSENİZ DEĞİŞTİRİN

print("="*70)
print("💾 SONUÇLARI GOOGLE DRIVE'A KAYDET")
print("="*70)

# Hedef klasörü oluştur
os.makedirs(SAVE_PATH, exist_ok=True)

# Tüm sonuçları kopyala
print(f"\n📁 Kayıt konumu: {SAVE_PATH}")
print(f"\n🔄 Dosyalar kopyalanıyor...")

# runs klasörünü kopyala
if results_dir.exists():
    dest_dir = Path(SAVE_PATH) / 'train'
    if dest_dir.exists():
        shutil.rmtree(dest_dir)
    shutil.copytree(results_dir, dest_dir)
    print("✅ Eğitim sonuçları kopyalandı")

# Dosya özeti
print(f"\n📊 Kaydedilen Dosyalar:")
important_files = [
    'weights/best.pt',
    'weights/last.pt',
    'weights/best.onnx',
    'weights/best.torchscript',
    'results.png',
    'confusion_matrix.png',
    'F1_curve.png',
    'PR_curve.png'
]

for file in important_files:
    file_path = dest_dir / file
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024**2)
        print(f"   ✅ {file:30s} ({size_mb:.1f} MB)")

print(f"\n🎉 Tüm sonuçlar Google Drive'a kaydedildi!")
print(f"📂 Konum: {SAVE_PATH}")

print("\n" + "="*70)

## 1️⃣6️⃣ Final Özet ve Öneriler

In [ ]:
print("="*70)
print("🎉 EĞİTİM TAMAMLANDI - FİNAL ÖZET")
print("="*70)

if best_model_path.exists():
    # Model bilgileri
    model_size = best_model_path.stat().st_size / (1024**2)
    
    print(f"\n📦 Model Bilgileri:")
    print(f"   • Model: YOLOv8n")
    print(f"   • Dosya Boyutu: {model_size:.1f} MB")
    print(f"   • Sınıf Sayısı: {data_config['nc']}")
    print(f"   • Eğitim Epoch: {EPOCHS}")
    print(f"   • Batch Size: {BATCH_SIZE}")
    print(f"   • Image Size: {IMG_SIZE}")
    
    print(f"\n🎯 Performans:")
    if 'metrics' in locals():
        print(f"   • mAP50:     {metrics.box.map50:.4f}")
        print(f"   • mAP50-95:  {metrics.box.map:.4f}")
        print(f"   • Precision: {metrics.box.mp:.4f}")
        print(f"   • Recall:    {metrics.box.mr:.4f}")
    
    print(f"\n💾 Kaydedilen Dosyalar:")
    print(f"   • Best Model: {SAVE_PATH}/train/weights/best.pt")
    print(f"   • ONNX Model: {SAVE_PATH}/train/weights/best.onnx")
    print(f"   • TorchScript: {SAVE_PATH}/train/weights/best.torchscript")
    print(f"   • Tüm Grafikler: {SAVE_PATH}/train/")
    
    print(f"\n🚀 Sonraki Adımlar:")
    print(f"   1. Confusion matrix'i inceleyin (nadir sınıfların performansı)")
    print(f"   2. Yanlış tahminleri analiz edin")
    print(f"   3. Gerekirse daha fazla epoch ile fine-tune yapın")
    print(f"   4. Gerçek test görsellerinde deneyin")
    print(f"   5. Model performansından memnunsanız deployment yapın")
    
    print(f"\n💡 Model Kullanımı:")
    print(f"   ```python")
    print(f"   from ultralytics import YOLO")
    print(f"   model = YOLO('{SAVE_PATH}/train/weights/best.pt')")
    print(f"   results = model.predict('image.jpg')")
    print(f"   ```")
    
    print(f"\n✅ Veri Dengesizliği Çözümü:")
    print(f"   • Class loss gain artırıldı")
    print(f"   • MixUp & Copy-Paste eklendi")
    print(f"   • Güçlendirilmiş augmentation kullanıldı")
    print(f"   → Nadir sınıfların performansı iyileştirildi")

print("\n" + "="*70)
print("🎊 BAŞARILAR! Model eğitimi tamamlandı.")
print("="*70)

---

## 📚 Ek Notlar:

### 🔄 Yeniden Eğitim:
```python
# Kaldığınız yerden devam etmek için:
model = YOLO('/content/runs/detect/train/weights/last.pt')
model.train(resume=True)
```

### 📊 Farklı Model Boyutları:
```python
# Daha büyük model için:
model = YOLO('yolov8s.pt')  # Small
model = YOLO('yolov8m.pt')  # Medium
model = YOLO('yolov8l.pt')  # Large
```

### 🎯 Hyperparameter Tuning:
```python
# Learning rate değiştirme:
model.train(lr0=0.005, lrf=0.01)

# Daha fazla epoch:
model.train(epochs=200)

# Farklı batch size:
model.train(batch=64)  # T4 GPU için maksimum
```

---

### 📞 Destek:
- YOLOv8 Dokümantasyon: https://docs.ultralytics.com
- GitHub Issues: https://github.com/ultralytics/ultralytics/issues

---

**🎉 İyi Eğitimler!**